# 📰 audeMAG — Analyse textuelle des magazines + croisement enquête

Ce notebook fait **deux choses principales** :
1. Analyser le texte extrait des magazines (quels sujets, quels mots, quels lieux, quelles personnes)
2. Croiser ces résultats avec les **attentes des lecteurs** issues de l'enquête de satisfaction

**Il produit 9 graphiques + 5 fichiers CSV/JSON exportables.**

---
> **Comment utiliser ce notebook :**
> - Lance les cellules **dans l'ordre** (Ctrl+Enter ou le bouton ▶)
> - La cellule 1 installe les dépendances (~2 min la première fois)
> - La cellule 2 te demande d'uploader ton fichier texte extrait du PDF
> - Modifie uniquement la **cellule de configuration** (cellule 3)


## Cellule 1 — Installation des dépendances
*À lancer une seule fois. Prend ~2 minutes.*

In [ ]:
!pip install -q spacy matplotlib wordcloud pandas seaborn
!python -m spacy download fr_core_news_lg
print("✅ Installation terminée")

## Cellule 2 — Upload du fichier ZIP

Upload ton fichier `.zip` contenant tous tes magazines en `.txt`.

Le script va :
1. Dézipper le contenu dans `magazines_txt/`
2. Lire la page 1 de chaque fichier pour extraire la date et le numero
3. Renommer chaque fichier au format `AUDEMAG-AAAA-MM.txt`

Les graphiques seront ensuite tries chronologiquement.


In [ ]:
from google.colab import files as colab_files
import os, re, zipfile
from pathlib import Path

# Upload
print('📦 Selectionne ton fichier ZIP contenant les magazines :')
uploaded = colab_files.upload()

ZIP_UPLOADE = list(uploaded.keys())[0]
print(f'\n✅ Fichier ZIP charge : {ZIP_UPLOADE}')
print(f'   Taille : {os.path.getsize(ZIP_UPLOADE):,} octets\n')

# Extraction
EXTRACT_DIR = Path('magazines_txt')
EXTRACT_DIR.mkdir(exist_ok=True)

with zipfile.ZipFile(ZIP_UPLOADE, 'r') as zf:
    txt_files = [f for f in zf.namelist()
                 if f.endswith('.txt') and not f.startswith('__')]
    for f in txt_files:
        data = zf.read(f)
        (EXTRACT_DIR / Path(f).name).write_bytes(data)

print(f'📂 {len(txt_files)} fichier(s) extrait(s)\n')

# Mapping mois FR -> numero
MOIS_MAP = {
    'JANVIER':'01','FEVRIER':'02','MARS':'03',
    'AVRIL':'04','MAI':'05','JUIN':'06','JUILLET':'07',
    'AOUT':'08','SEPTEMBRE':'09','OCTOBRE':'10',
    'NOVEMBRE':'11','DECEMBRE':'12',
}

def normaliser(s):
    """Enleve les accents pour simplifier la recherche."""
    rempl = {'É':'E','È':'E','Ê':'E','Ë':'E','À':'A','Â':'A',
             'Î':'I','Ï':'I','Ô':'O','Û':'U','Ù':'U','Ü':'U','Ç':'C'}
    s = s.upper()
    for k, v in rempl.items():
        s = s.replace(k, v)
    return s

def extraire_date(texte):
    m = re.search(r'={10,}\s*PAGE\s*1\s*={10,}(.*?)(?:={10,}|$)', texte, re.DOTALL)
    page1 = m.group(1) if m else texte[:1200]
    page1_norm = normaliser(page1)

    # Numero du magazine
    nm = re.search(r'(?:^|\n)\s*(\d{1,3})\s*\n\s*#', page1)
    if not nm:
        nm = re.search(r'(?:^|\n)\s*(\d{1,3})\s*\n', page1)
    numero = nm.group(1).zfill(2) if nm else '??'

    # Annee
    am = re.search(r'\b(20\d{2})\b', page1)
    annee = am.group(1) if am else '????'

    # Mois
    mois_trouves = []
    for mois_fr, num_m in MOIS_MAP.items():
        if mois_fr in page1_norm:
            pos = page1_norm.find(mois_fr)
            mois_trouves.append((pos, num_m, mois_fr))
    mois_trouves.sort()

    if mois_trouves:
        mois_num = mois_trouves[0][1]
        # Recuperer le label avec les deux mois si bimestriel
        pat = r'([A-ZAEIOUFEVRILUJSOND]+(?:[/\-][A-ZAEIOUFEVRILUJSOND]+)*)\s*\n\s*' + annee
        lm = re.search(pat, page1_norm)
        label_mois = lm.group(1) if lm else mois_trouves[0][2]
    else:
        mois_num = '00'
        label_mois = 'INCONNU'

    nom = f'AUDEMAG-{annee}-{mois_num}'
    date_sort = f'{annee}-{mois_num}'
    label = f'N{numero} {label_mois} {annee}'
    return nom, date_sort, label, numero

# Renommage
print('🔍 Detection des dates et renommage...\n')
CATALOGUE = {}
doublons = {}

for fichier in sorted(EXTRACT_DIR.glob('*.txt')):
    texte = fichier.read_text(encoding='utf-8', errors='ignore')
    nom, date_sort, label, numero = extraire_date(texte)

    # Gerer les doublons de mois
    if nom in CATALOGUE or nom in doublons:
        doublons[nom] = doublons.get(nom, 1) + 1
        nom_final = f'{nom}-v{doublons[nom]}'
    else:
        nom_final = nom

    nouveau_chemin = EXTRACT_DIR / f'{nom_final}.txt'
    if fichier != nouveau_chemin:
        fichier.rename(nouveau_chemin)

    CATALOGUE[nom_final] = {
        'date_sortable': date_sort,
        'label': label,
        'numero': numero,
        'chemin': nouveau_chemin,
    }
    print(f'  {fichier.name:<35} ->  {nom_final}.txt   ({label})')

print(f'\n✅ {len(CATALOGUE)} magazine(s) renomme(s) et prets a analyser')
print('\n📅 Ordre chronologique :')
for k, v in sorted(CATALOGUE.items(), key=lambda x: x[1]['date_sortable']):
    print(f'   {v["date_sortable"]}  {v["label"]}')

## Cellule 3 — Configuration
Rien a modifier — le dossier `magazines_txt/` et le catalogue des dates
ont ete crees automatiquement par la cellule 2.


In [ ]:
from pathlib import Path

DOSSIER_MAGAZINES = 'magazines_txt'
DOSSIER_SORTIE    = Path('resultats_analyse')
DOSSIER_SORTIE.mkdir(exist_ok=True)
MODELE_SPACY = 'fr_core_news_lg'

fichiers = sorted(Path(DOSSIER_MAGAZINES).glob('*.txt'))
print(f'✅ {len(fichiers)} magazine(s) detecte(s)')
print(f'   Dossier sortie : {DOSSIER_SORTIE.resolve()}')

## Cellule 4 — Données de l'enquête (146 répondants)
Ces données ont été extraites du fichier Excel de l'enquête.  
Elles servent pour les **croisements** entre contenu des magazines et attentes des lecteurs.


In [ ]:
# Rubriques lues par les répondants (colonnes N, O, P de l'enquête)
RUBRIQUES_ENQUETE = {
    "En bref (actualités)": 127,
    "Par ici les sorties (agenda)": 104,
    "Dossiers thématiques": 98,
    "L'Art d'être Audois (portraits)": 88,
    "Manger audois": 79,
    "Focus (initiatives locales)": 71,
    "C'est pratique (démarches)": 62,
    "Quoi de neuf sur les routes ?": 47,
    "Dans la peau de (agents)": 43,
    "Tribune politique": 43,
    "Ça c'est du sport": 18,
    "Grand angle (photos)": 18,
    "Chronique occitane": 11,
}

# Priorités de refonte (colonne AE — 3 choix possibles par répondant)
PRIORITES_REFONTE = {
    "Plus de contenu local par canton/commune": 112,
    "Rubrique services pratiques renforcée": 60,
    "Plus de photos et infographies": 30,
    "Moins de textes institutionnels, plus de pédagogie": 29,
    "Versions numériques enrichies": 13,
}

# Sujets souhaités (colonne R — réponses libres regroupées en catégories)
# Chaque catégorie est associée à des mots-clés cherchés dans le texte du magazine
SUJETS_SOUHAITES_MOTS_CLES = {
    "communes rurales / villages": ["commune", "village", "rural", "local", "canton",
                                    "bourg", "hameau", "localité"],
    "environnement / nature":      ["environnement", "nature", "écologie", "ENS",
                                    "faune", "flore", "biodiversité", "forêt",
                                    "arbre", "plante", "herbe"],
    "histoire / patrimoine":       ["histoire", "historique", "patrimoine", "château",
                                    "monument", "tradition", "mémoire"],
    "emploi / économie locale":    ["emploi", "travail", "entreprise", "économie",
                                    "artisan", "commerce", "circuit court"],
    "terroir / gastronomie":       ["terroir", "recette", "cuisine", "produit",
                                    "viticulture", "vigne", "vin", "marché"],
    "jeunesse / éducation":        ["jeune", "enfant", "ado", "adolescent", "école",
                                    "collège", "lycée", "CMJ"],
    "personnes âgées / santé":     ["âgé", "senior", "santé", "médecin", "aide",
                                    "soins", "retraite", "EHPAD"],
    "culture / arts":              ["culture", "culturel", "festival", "artiste",
                                    "exposition", "musique", "spectacle"],
    "mobilité / transport":        ["transport", "mobilité", "route", "bus",
                                    "déplacement", "piste cyclable", "vélo"],
}

# Mots-clés thématiques pour mesurer la présence des rubriques dans le texte
RUBRIQUES_MOTS_CLES = {
    "portraits / initiatives locales": ["portrait", "initiative", "habitant",
                                        "personnalité", "témoignage", "rencontre"],
    "contenu local":                   ["commune", "village", "canton", "mairie",
                                        "local", "territoire", "quartier"],
    "services pratiques":              ["démarche", "contact", "pratique", "service",
                                        "aide", "formulaire", "numéro", "adresse"],
    "sport":                           ["sport", "sportif", "championnat", "club",
                                        "équipe", "compétition", "athlète"],
    "culture / agenda":                ["sortie", "agenda", "festival", "exposition",
                                        "concert", "spectacle", "musée"],
    "institutionnel":                  ["conseil", "élu", "budget", "vote",
                                        "délibération", "assemblée", "politique"],
    "environnement":                   ["environnement", "nature", "biodiversité",
                                        "forêt", "eau", "écologie", "transition"],
    "gastronomie / terroir":           ["recette", "produit", "terroir", "marché",
                                        "artisan", "agriculture", "vigne"],
}

# Mots vides supplémentaires propres au corpus audeMAG
STOPWORDS_CUSTOM = {
    "département", "aude", "audois", "audoise", "audoises", "audeMAG",
    "conseil", "départemental", "collectivité", "territoire",
    "plus", "très", "aussi", "ainsi", "tout", "tous", "lors", "dont",
    "cette", "avoir", "être", "faire", "aller", "pouvoir", "vouloir",
    "notamment", "afin", "permet", "permettre", "mettre",
    "numéro", "magazine", "audemag", "page", "http", "www",
    "après", "avant", "sous", "sur", "dans", "avec", "pour", "par",
}

print("✅ Données de l'enquête chargées")

## Cellule 5 — Imports et chargement de spaCy

In [ ]:
import re
import json
import warnings
from collections import Counter, defaultdict

import spacy
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from wordcloud import WordCloud

warnings.filterwarnings("ignore")

print("⏳ Chargement du modèle spaCy fr_core_news_lg...")
nlp = spacy.load(MODELE_SPACY)
nlp.Defaults.stop_words |= STOPWORDS_CUSTOM
print("✅ Modèle spaCy chargé et prêt")

## Cellule 6 — Fonctions d'analyse
Trois fonctions :
- `split_pages()` : découpe le texte en pages selon les marqueurs `PAGE N`
- `analyser_corpus()` : fait tourner spaCy, extrait les mots, entités et scores thématiques
- `score_sujets_souhaites()` : mesure la présence des sujets demandés par les lecteurs


In [ ]:
def split_pages(texte):
    """Découpe le texte en pages selon les marqueurs == PAGE N =="""
    blocs = re.split(r"={10,}\s*PAGE\s*(\d+)\s*={10,}", texte)
    pages = []
    for i in range(1, len(blocs) - 1, 2):
        pages.append({"page": int(blocs[i]), "texte": blocs[i + 1].strip()})
    return pages


def analyser_corpus(pages, nom_mag="corpus"):
    """
    Passe spaCy sur toutes les pages du magazine.
    Retourne : fréquences des mots, entités nommées, scores thématiques par page.
    """
    print(f"  ⏳ Analyse NLP de '{nom_mag}' ({len(pages)} pages)...")

    docs = list(nlp.pipe(
        [p["texte"] for p in pages],
        batch_size=10,
        disable=["parser"],
    ))

    all_lemmes, all_ents, scores_pages = [], [], []

    for page_info, doc in zip(pages, docs):
        # Garder uniquement noms, verbes, adjectifs — pas les mots grammaticaux
        lemmes_page = [
            token.lemma_.lower()
            for token in doc
            if not token.is_stop
            and not token.is_punct
            and not token.is_space
            and not token.like_url
            and len(token.text) > 2
            and token.pos_ in {"NOUN", "VERB", "ADJ", "PROPN"}
        ]
        all_lemmes.extend(lemmes_page)

        # Entités nommées : personnes, lieux, organisations
        for ent in doc.ents:
            if ent.label_ in {"PER", "ORG", "LOC", "GPE"}:
                all_ents.append({
                    "magazine": nom_mag,
                    "page": page_info["page"],
                    "texte": ent.text.strip(),
                    "type": ent.label_,
                })

        # Score thématique de la page (présence des mots-clés)
        lemmes_set = set(lemmes_page)
        scores = {"magazine": nom_mag, "page": page_info["page"]}
        for theme, mots in RUBRIQUES_MOTS_CLES.items():
            scores[theme] = sum(1 for m in mots if m in lemmes_set)
        scores_pages.append(scores)

    freq_lemmes = Counter(all_lemmes)
    print(f"  ✅ {len(all_lemmes):,} tokens | {len(freq_lemmes):,} lemmes uniques | {len(all_ents):,} entités\n")

    return {
        "nom": nom_mag,
        "n_pages": len(pages),
        "freq_lemmes": freq_lemmes,
        "entites": all_ents,
        "scores_pages": pd.DataFrame(scores_pages),
    }


def score_sujets_souhaites(freq_lemmes):
    """Mesure la présence dans le corpus des sujets demandés par les lecteurs."""
    return {
        sujet: sum(freq_lemmes.get(m, 0) for m in mots)
        for sujet, mots in SUJETS_SOUHAITES_MOTS_CLES.items()
    }


print("✅ Fonctions définies")

## Cellule 7 — Verification du catalogue chronologique
Verifie que toutes les dates ont ete bien detectees.
Si une date est incorrecte, corrige manuellement le dictionnaire `CATALOGUE` dans le bloc du bas.


In [ ]:
print('📅 Catalogue des magazines\n')
print(f'{"Fichier":<30} {"Date":<12} {"Label"}')
print('-' * 65)
for k, v in sorted(CATALOGUE.items(), key=lambda x: x[1]['date_sortable']):
    print(f'{k:<30} {v["date_sortable"]:<12} {v["label"]}')
print(f'\n✅ {len(CATALOGUE)} numero(s) au total')

# Correction manuelle si une date est mal lue :
# CATALOGUE['AUDEMAG-2023-00']['date_sortable'] = '2023-03'
# CATALOGUE['AUDEMAG-2023-00']['label']         = 'N42 MARS 2023'

## Cellule 7 — Analyse des magazines ⏳
**C'est la cellule la plus longue (~30 sec à 2 min selon la taille du corpus).**  
Elle fait tourner spaCy sur tout le texte.

In [ ]:
resultats_magazines = []

if DOSSIER_MAGAZINES and Path(DOSSIER_MAGAZINES).exists():
    # Mode multi-fichiers (64 magazines)
    fichiers = sorted(Path(DOSSIER_MAGAZINES).glob("*.txt"))
    print(f"📂 {len(fichiers)} magazine(s) trouvé(s) dans '{DOSSIER_MAGAZINES}'\n")
    for f in fichiers:
        texte = f.read_text(encoding="utf-8")
        pages = split_pages(texte)
        r = analyser_corpus(pages, nom_mag=f.stem)
        resultats_magazines.append(r)
else:
    # Mode fichier unique
    print(f"📄 Chargement de '{FICHIER_TEXTE_UNIQUE}'...")
    texte = Path(FICHIER_TEXTE_UNIQUE).read_text(encoding="utf-8")
    pages = split_pages(texte)
    print(f"   → {len(pages)} pages détectées\n")
    r = analyser_corpus(pages, nom_mag=Path(FICHIER_TEXTE_UNIQUE).stem)
    resultats_magazines.append(r)

# Fusion en corpus global
freq_globale = Counter()
entites_globales = []
for r in resultats_magazines:
    freq_globale += r["freq_lemmes"]
    entites_globales.extend(r["entites"])

df_entites = pd.DataFrame(entites_globales)
df_scores  = pd.concat([r["scores_pages"] for r in resultats_magazines], ignore_index=True)
freq_ents  = Counter(df_entites["texte"].tolist()) if not df_entites.empty else Counter()

# Croisements
scores_croisement = score_sujets_souhaites(freq_globale)
scores_rubriques_mag = {t: df_scores[t].sum() for t in RUBRIQUES_MOTS_CLES}

print(f"\n📊 CORPUS TOTAL")
print(f"   {sum(freq_globale.values()):,} mots porteurs de sens")
print(f"   {len(freq_globale):,} lemmes uniques")
print(f"   {len(df_entites):,} entités nommées détectées")
print(f"\n🔝 Top 15 mots :")
for mot, n in freq_globale.most_common(15):
    print(f"   {mot:<25} {n:>4}")

## Cellule 8 — Visualisation 1 : Nuage de mots

In [ ]:
plt.style.use("seaborn-v0_8-whitegrid")
BLEU, BLEU2, VERT, ORANGE, ROUGE, GRIS = "#185FA5","#378ADD","#1D9E75","#EF9F27","#E24B4A","#888780"
PALETTE = [BLEU, BLEU2, VERT, ORANGE, ROUGE, "#D4537E", GRIS, "#9FE1CB"]

fig, ax = plt.subplots(figsize=(14, 7))
wc = WordCloud(
    width=1400, height=700, background_color="white",
    colormap="Blues", max_words=120, prefer_horizontal=0.8,
).generate_from_frequencies(dict(freq_globale.most_common(150)))
ax.imshow(wc, interpolation="bilinear")
ax.axis("off")
ax.set_title("Nuage de mots — corpus audeMAG", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.savefig(DOSSIER_SORTIE / "01_nuage_mots.png", dpi=150, bbox_inches="tight")
plt.show()
print("💾 Sauvegardé : 01_nuage_mots.png")

## Cellule 9 — Visualisation 2 : Top 30 mots fréquents

In [ ]:
top30 = pd.DataFrame(freq_globale.most_common(30), columns=["mot", "freq"])
fig, ax = plt.subplots(figsize=(11, 9))
bars = ax.barh(top30["mot"][::-1], top30["freq"][::-1],
               color=[BLEU if i < 10 else BLEU2 if i < 20 else GRIS for i in range(30)])
ax.set_xlabel("Nombre d'occurrences", fontsize=12)
ax.set_title("Top 30 mots les plus fréquents\n(noms, verbes, adjectifs — stopwords retirés)",
             fontsize=13, fontweight="bold")
for bar, val in zip(bars, top30["freq"][::-1]):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            str(val), va="center", fontsize=8)
plt.tight_layout()
plt.savefig(DOSSIER_SORTIE / "02_top30_mots.png", dpi=150, bbox_inches="tight")
plt.show()
print("💾 Sauvegardé : 02_top30_mots.png")

## Cellule 10 — Visualisation 3 : Couverture thématique

In [ ]:
theme_totaux = pd.Series({t: df_scores[t].sum() for t in RUBRIQUES_MOTS_CLES}).sort_values()
fig, ax = plt.subplots(figsize=(10, 6))
colors_t = [PALETTE[i % len(PALETTE)] for i in range(len(theme_totaux))]
bars = ax.barh(theme_totaux.index, theme_totaux.values, color=colors_t)
ax.set_xlabel("Score (occurrences de mots-clés par thème)", fontsize=12)
ax.set_title("Couverture thématique dans les magazines", fontsize=13, fontweight="bold")
for bar, val in zip(bars, theme_totaux.values):
    ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
            str(int(val)), va="center", fontsize=10)
plt.tight_layout()
plt.savefig(DOSSIER_SORTIE / "03_themes_magazines.png", dpi=150, bbox_inches="tight")
plt.show()
print("💾 Sauvegardé : 03_themes_magazines.png")

## Cellule 11 — Visualisation 4 : Heatmap thématique par page

In [ ]:
theme_cols = list(RUBRIQUES_MOTS_CLES.keys())
if len(resultats_magazines) > 1:
    df_hm = df_scores.groupby("magazine")[theme_cols].sum()
    # Tri chronologique
    ordre = [r["nom"] for r in resultats_magazines]
    df_hm = df_hm.reindex([o for o in ordre if o in df_hm.index])
    ylabel = "Magazine (ordre chronologique)"
else:
    df_hm = df_scores.set_index("page")[theme_cols]
    ylabel = "Page"

fig_h = max(6, len(df_hm) * 0.4 + 2)
fig, ax = plt.subplots(figsize=(13, fig_h))
sns.heatmap(df_hm.T, ax=ax, cmap="YlOrRd", linewidths=0.3, linecolor="white",
            cbar_kws={"label": "Score thématique"})
ax.set_title("Carte de chaleur — où parle-t-on de quoi ?", fontsize=13, fontweight="bold")
ax.set_xlabel(ylabel)
plt.tight_layout()
plt.savefig(DOSSIER_SORTIE / "04_heatmap_themes.png", dpi=150, bbox_inches="tight")
plt.show()
print("💾 Sauvegardé : 04_heatmap_themes.png")

## Cellule 12 — Visualisation 5 : Entités nommées (personnes, lieux, orgs)

In [ ]:
if not df_entites.empty:
    fig, axes = plt.subplots(1, 3, figsize=(16, 7))
    for ax, (etype, label, color) in zip(
        axes,
        [("PER","Personnes",ORANGE), ("LOC","Lieux / communes",VERT), ("ORG","Organisations",BLEU2)]
    ):
        subset = df_entites[df_entites["type"] == etype]["texte"]
        top = pd.DataFrame(Counter(subset).most_common(12), columns=["entité","n"])
        if top.empty:
            ax.axis("off"); continue
        ax.barh(top["entité"][::-1], top["n"][::-1], color=color)
        ax.set_title(label, fontsize=12, fontweight="bold")
        ax.set_xlabel("Occurrences")
        for bar, val in zip(ax.patches, top["n"][::-1]):
            ax.text(bar.get_width()+0.1, bar.get_y()+bar.get_height()/2,
                    str(val), va="center", fontsize=8)
    plt.suptitle("Top entités nommées dans les magazines", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(DOSSIER_SORTIE / "05_entites_nommees.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("💾 Sauvegardé : 05_entites_nommees.png")
else:
    print("⚠️ Aucune entité détectée dans ce corpus")

## Cellule 13 — Visualisation 6 : ⭐ Croisement rubriques enquête vs magazines
**C'est le graphique central du projet.** Deux barres côte à côte pour chaque rubrique :
- Bleue = combien de lecteurs disent lire cette rubrique (enquête)
- Orange = à quel point cette rubrique est présente dans le magazine (NLP)


In [ ]:
CORRESPONDANCES = {
    "En bref (actualités)":            None,
    "Par ici les sorties (agenda)":    "culture / agenda",
    "Dossiers thématiques":            None,
    "L'Art d'être Audois (portraits)": "portraits / initiatives locales",
    "Manger audois":                   "gastronomie / terroir",
    "Focus (initiatives locales)":     "portraits / initiatives locales",
    "C'est pratique (démarches)":      "services pratiques",
    "Quoi de neuf sur les routes ?":   "mobilité / transport",
    "Dans la peau de (agents)":        "institutionnel",
    "Tribune politique":               "institutionnel",
    "Ça c'est du sport":               "sport",
    "Grand angle (photos)":            None,
    "Chronique occitane":              None,
}

fig, ax = plt.subplots(figsize=(13, 7))
rubriques_labels = list(RUBRIQUES_ENQUETE.keys())
x = range(len(rubriques_labels))
width = 0.4

ax.bar([i - width/2 for i in x],
       [RUBRIQUES_ENQUETE[r] for r in rubriques_labels],
       width=width, color=BLEU, label="Lecteurs qui lisent cette rubrique (enquête)", alpha=0.9)

max_enquete = max(RUBRIQUES_ENQUETE.values())
scores_mag_bar = []
for rub in rubriques_labels:
    theme_mag = CORRESPONDANCES.get(rub)
    if theme_mag and theme_mag in scores_rubriques_mag:
        val_norm = min(scores_rubriques_mag[theme_mag] / max(scores_rubriques_mag.values()) * max_enquete, max_enquete)
        scores_mag_bar.append(val_norm)
    else:
        scores_mag_bar.append(0)

ax.bar([i + width/2 for i in x], scores_mag_bar,
       width=width, color=ORANGE, label="Présence dans le magazine (score NLP normalisé)", alpha=0.9)

ax.set_xticks(list(x))
ax.set_xticklabels(rubriques_labels, rotation=35, ha="right", fontsize=9)
ax.set_ylabel("Score")
ax.set_title("Rubriques : ce que lisent les lecteurs vs présence dans les magazines",
             fontsize=12, fontweight="bold")
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig(DOSSIER_SORTIE / "06_croisement_rubriques.png", dpi=150, bbox_inches="tight")
plt.show()
print("💾 Sauvegardé : 06_croisement_rubriques.png")

## Cellule 14 — Visualisation 7 : ⭐ Sujets souhaités vs présence dans les magazines
Répond à la question : **les sujets demandés par les lecteurs sont-ils déjà traités ?**
- 🔴 Rouge = peu couvert → priorité pour la refonte
- 🟡 Orange = couverture moyenne
- 🟢 Vert = déjà bien traité


In [ ]:
sujets_labels = list(SUJETS_SOUHAITES_MOTS_CLES.keys())
scores_mag_sujets = [scores_croisement[s] for s in sujets_labels]
max_mag = max(scores_mag_sujets) if max(scores_mag_sujets) > 0 else 1

colors_bar = [
    VERT if scores_croisement[s] >= max_mag * 0.6
    else ORANGE if scores_croisement[s] >= max_mag * 0.3
    else ROUGE
    for s in sujets_labels
]

fig, ax = plt.subplots(figsize=(13, 7))
bars = ax.barh(sujets_labels[::-1],
               [scores_croisement[s] for s in sujets_labels[::-1]],
               color=colors_bar[::-1])

patches = [
    mpatches.Patch(color=VERT,   label="Bien couvert dans le magazine"),
    mpatches.Patch(color=ORANGE, label="Couverture moyenne"),
    mpatches.Patch(color=ROUGE,  label="Peu couvert — priorité potentielle"),
]
ax.legend(handles=patches, fontsize=10, loc="lower right")
ax.set_xlabel("Score de présence dans le magazine (occurrences de mots-clés)")
ax.set_title("Sujets demandés par les lecteurs : sont-ils déjà traités ?",
             fontsize=12, fontweight="bold")
for bar, val in zip(bars, [scores_croisement[s] for s in sujets_labels[::-1]]):
    ax.text(bar.get_width()+0.3, bar.get_y()+bar.get_height()/2,
            str(int(val)), va="center", fontsize=9)
plt.tight_layout()
plt.savefig(DOSSIER_SORTIE / "07_sujets_souhaites_vs_magazines.png", dpi=150, bbox_inches="tight")
plt.show()
print("💾 Sauvegardé : 07_sujets_souhaites_vs_magazines.png")

## Cellule 15 — Visualisation 8 : Priorités de refonte (rappel enquête)

In [ ]:
pr = pd.Series(PRIORITES_REFONTE).sort_values()
colors_pr = [ROUGE if v == pr.max() else ORANGE if v >= pr.max()*0.4 else BLEU2 for v in pr.values]
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(pr.index, pr.values, color=colors_pr)
ax.set_xlabel("Nombre de répondants (146 au total, 3 choix possibles)")
ax.set_title("Ce que les lecteurs veulent changer en priorité", fontsize=12, fontweight="bold")
for bar, val in zip(bars, pr.values):
    ax.text(bar.get_width()+0.5, bar.get_y()+bar.get_height()/2,
            str(int(val)), va="center", fontsize=10, fontweight="bold")
plt.tight_layout()
plt.savefig(DOSSIER_SORTIE / "08_priorites_refonte.png", dpi=150, bbox_inches="tight")
plt.show()
print("💾 Sauvegardé : 08_priorites_refonte.png")

## Cellule 16 — Visualisation 9 : Évolution thématique (mode multi-magazines)
**Cette cellule ne produit quelque chose que si tu analyses plusieurs magazines.**  
Elle montre comment la répartition des thèmes évolue d'un numéro à l'autre.


In [ ]:
if len(resultats_magazines) > 1:
    df_evol = df_scores.groupby("magazine")[theme_cols].sum()
    # Tri chronologique
    ordre = [r["nom"] for r in resultats_magazines]
    df_evol = df_evol.reindex([o for o in ordre if o in df_evol.index])
    df_evol_norm = df_evol.div(df_evol.sum(axis=1), axis=0) * 100
    fig, ax = plt.subplots(figsize=(14, 6))
    df_evol_norm.plot(kind="bar", stacked=True, ax=ax, colormap="tab10", width=0.8)
    ax.set_xlabel("Numéro du magazine")
    ax.set_ylabel("Part thématique (%)")
    ax.set_title("Évolution de la répartition thématique d'un numéro à l'autre",
                 fontsize=13, fontweight="bold")
    ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=9)
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(DOSSIER_SORTIE / "09_evolution_thematique.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("💾 Sauvegardé : 09_evolution_thematique.png")
else:
    print("ℹ️ Cette visualisation nécessite plusieurs magazines.")
    print("   → Mets tes fichiers .txt dans un dossier 'magazines_txt/'")
    print("   → Change DOSSIER_MAGAZINES = 'magazines_txt' dans la cellule 3")

## Cellule 17 — Export des fichiers CSV et JSON

In [ ]:
print("💾 Export des données...\n")

pd.DataFrame(freq_globale.most_common(300), columns=["lemme","fréquence"]).to_csv(
    DOSSIER_SORTIE / "mots_frequents.csv", index=False, encoding="utf-8-sig")
print("  ✅ mots_frequents.csv")

if not df_entites.empty:
    df_entites.to_csv(DOSSIER_SORTIE / "entites_nommees.csv", index=False, encoding="utf-8-sig")
    print("  ✅ entites_nommees.csv")

df_scores.to_csv(DOSSIER_SORTIE / "themes_par_page.csv", index=False, encoding="utf-8-sig")
print("  ✅ themes_par_page.csv")

max_crois = max(scores_croisement.values()) or 1
df_crois = pd.DataFrame([{
    "sujet_souhaité_par_lecteurs": s,
    "score_présence_magazine": scores_croisement[s],
    "mots_clés_utilisés": ", ".join(SUJETS_SOUHAITES_MOTS_CLES[s]),
    "niveau": ("Bien couvert" if scores_croisement[s] >= max_crois*0.6
               else "Couverture moyenne" if scores_croisement[s] >= max_crois*0.3
               else "Peu couvert"),
} for s in SUJETS_SOUHAITES_MOTS_CLES]).sort_values("score_présence_magazine", ascending=False)
df_crois.to_csv(DOSSIER_SORTIE / "croisement_sujets_vs_magazines.csv", index=False, encoding="utf-8-sig")
print("  ✅ croisement_sujets_vs_magazines.csv")

resume = {
    "n_magazines_analysés": len(resultats_magazines),
    "n_tokens_total": sum(freq_globale.values()),
    "n_lemmes_uniques": len(freq_globale),
    "top_50_mots": freq_globale.most_common(50),
    "top_entites": freq_ents.most_common(30),
    "scores_croisement": scores_croisement,
    "scores_rubriques_magazines": {k: int(v) for k, v in scores_rubriques_mag.items()},
}
with open(DOSSIER_SORTIE / "resume_analyse.json", "w", encoding="utf-8") as f:
    json.dump(resume, f, ensure_ascii=False, indent=2)
print("  ✅ resume_analyse.json")

print(f"\n📁 Tous les fichiers sont dans : {DOSSIER_SORTIE.resolve()}/")

## Cellule 18 — Télécharger les résultats depuis Colab

In [ ]:
from google.colab import files as colab_files
import zipfile

# Zipper tous les résultats
zip_path = "resultats_audemag.zip"
with zipfile.ZipFile(zip_path, "w") as zf:
    for f in DOSSIER_SORTIE.iterdir():
        zf.write(f, f.name)

print(f"✅ Archive créée : {zip_path}")
print("⬇️  Téléchargement en cours...")
colab_files.download(zip_path)